# Data Exploration & Analysis

Explore the flood dataset:
- Load and examine data
- Check for missing values
- Analyze distributions
- Explore correlations
- Visualize relationships

## Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Add project to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from src import config

print("✓ Imports complete")

## 1. Load Data

In [ ]:
# Load the dataset
df = pd.read_csv(project_root / config.RAW_DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 10 rows:")
display(df.head(10))

## 2. Data Info

In [ ]:
print("Data Types:")
print(df.dtypes)

print(f"\nData Info:")
df.info()

print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

## 3. Missing Values

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})

print("Missing Values:")
print(missing_df[missing_df['Missing Count'] > 0])

if missing.sum() == 0:
    print("✓ No missing values found!")

## 4. Descriptive Statistics

In [ ]:
print("Descriptive Statistics:")
display(df.describe())

## 5. Target Variable Analysis

In [ ]:
# Analyze target variable
target_counts = df[config.TARGET_COLUMN].value_counts()
target_pct = df[config.TARGET_COLUMN].value_counts(normalize=True) * 100

print("Target Variable Distribution:")
print(target_counts)
print(f"\nPercentage:")
print(target_pct)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
target_counts.plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Flood Risk Distribution')
axes[0].set_xlabel('Flood Risk')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No Flood', 'Flood'], rotation=0)

# Pie chart
axes[1].pie(target_counts, labels=['No Flood', 'Flood'], autopct='%1.1f%%', 
            colors=['green', 'red'], startangle=90)
axes[1].set_title('Flood Risk Proportion')

plt.tight_layout()
plt.show()

## 6. Feature Distributions

In [ ]:
# Plot distributions for each feature
features = config.FEATURE_COLUMNS
fig, axes = plt.subplots(len(features), 1, figsize=(12, 4*len(features)))

if len(features) == 1:
    axes = [axes]

for idx, feature in enumerate(features):
    # Histogram with KDE
    df[feature].hist(bins=20, ax=axes[idx], edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {feature}')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequency')
    
    # Add statistics
    mean = df[feature].mean()
    median = df[feature].median()
    axes[idx].axvline(mean, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean:.2f}')
    axes[idx].axvline(median, color='green', linestyle='--', linewidth=2, label=f'Median: {median:.2f}')
    axes[idx].legend()

plt.tight_layout()
plt.show()

## 7. Correlation Analysis

In [ ]:
# Calculate correlation matrix
correlation_matrix = df.corr()

print("Correlation Matrix:")
print(correlation_matrix)

# Plot correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Correlation with target
print(f"\nCorrelation with {config.TARGET_COLUMN}:")
target_corr = correlation_matrix[config.TARGET_COLUMN].sort_values(ascending=False)
print(target_corr)

## 8. Feature vs Target

In [ ]:
# Plot features by target class
fig, axes = plt.subplots(len(features), 1, figsize=(12, 4*len(features)))

if len(features) == 1:
    axes = [axes]

for idx, feature in enumerate(features):
    # Box plot
    df.boxplot(column=feature, by=config.TARGET_COLUMN, ax=axes[idx])
    axes[idx].set_title(f'{feature} by Flood Risk')
    axes[idx].set_xlabel('Flood Risk')
    axes[idx].set_ylabel(feature)
    axes[idx].set_xticklabels(['No Flood', 'Flood'])
    
    # Remove the default title
    plt.sca(axes[idx])
    plt.xticks([1, 2], ['No Flood', 'Flood'])

plt.tight_layout()
plt.show()

## 9. Statistical Summary

In [ ]:
# Summary statistics by target class
print("Statistics for No Flood class:")
print(df[df[config.TARGET_COLUMN] == 0][features].describe())

print("\nStatistics for Flood class:")
print(df[df[config.TARGET_COLUMN] == 1][features].describe())

## Key Findings

Document your findings here:
- Data shape and structure
- Missing values
- Class balance
- Feature distributions
- Important correlations
- Any data quality issues